# 🚀 BDE Score: Interactive Stock Analysis Tutorial

**EU AI Act Compliant Stock Analysis API**

This interactive notebook demonstrates how to use BDE Score for transparent, explainable stock analysis.

## What You'll Learn
- How to analyze stocks using multi-factor scoring
- Understanding confidence levels and risk warnings
- Interpreting factor breakdowns
- Compliance features for EU AI Act

## 📊 Live Demo

Let's analyze some popular stocks and see the scoring system in action!

In [ ]:
# Install required packages
!pip install requests pandas matplotlib seaborn -q

import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Configure visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Environment ready!")

## 1. Analyze a Single Stock

Let's start by analyzing Tencent (HK.00700), one of the largest tech companies in Asia.

In [ ]:
# BDE Score API endpoint
API_BASE = "https://atlantic-remains-atomic-floor.trycloudflare.com"

def analyze_stock(stock_code: str, include_explanation: bool = True):
    """Analyze a stock using BDE Score API"""
    try:
        response = requests.post(
            f"{API_BASE}/api/score",
            json={
                "stock_code": stock_code,
                "include_explanation": include_explanation
            },
            timeout=10
        )
        
        if response.status_code == 200:
            return response.json()
        else:
            return {"error": f"API returned {response.status_code}", "details": response.text}
    except Exception as e:
        return {"error": str(e)}

# Analyze Tencent
print("🔍 Analyzing Tencent (HK.00700)...\n")
result = analyze_stock("HK.00700")

if "error" not in result:
    print(f"📈 Stock: {result['stock_code']}")
    print(f"⭐ BDE Score: {result['score']:.1f}/100")
    print(f"🎯 Confidence: {result['confidence']:.2%}")
    print(f"🕐 Timestamp: {result['timestamp']}")
    
    if 'explanation' in result and result['explanation']:
        print("\n📊 Factor Breakdown:")
        for factor, contribution in result['explanation']['factor_breakdown'].items():
            print(f"  • {factor}: {contribution:.1f}")
        
        if result['explanation'].get('risk_warnings'):
            print("\n⚠️  Risk Warnings:")
            for warning in result['explanation']['risk_warnings']:
                print(f"  • {warning}")
        
        print(f"\n✅ Human Review Required: {result['explanation']['human_review_required']}")
else:
    print(f"❌ Error: {result['error']}")
    print("\n💡 Note: The API may be temporarily unavailable. This is a demonstration of the analysis workflow.")

## 2. Batch Analysis: Compare Multiple Stocks

Let's compare several popular stocks across different markets.

In [ ]:
# Stocks to analyze
stocks_to_analyze = [
    ("HK.00700", "Tencent"),
    ("HK.09988", "Alibaba"),
    ("US.AAPL", "Apple"),
    ("US.MSFT", "Microsoft"),
    ("US.GOOG", "Google")
]

print("🔍 Batch analyzing 5 major tech stocks...\n")

results = []
for stock_code, name in stocks_to_analyze:
    print(f"Analyzing {name} ({stock_code})...")
    result = analyze_stock(stock_code, include_explanation=False)
    if "error" not in result:
        results.append({
            "Stock": name,
            "Code": stock_code,
            "BDE Score": result['score'],
            "Confidence": result['confidence']
        })

if results:
    # Create DataFrame
    df = pd.DataFrame(results)
    df = df.sort_values("BDE Score", ascending=False)
    
    print("\n📊 Analysis Results:")
    print(df.to_string(index=False))
    
    # Visualization
    plt.figure(figsize=(12, 6))
    bars = plt.barh(df['Stock'], df['BDE Score'], color='#2ecc71')
    plt.xlabel('BDE Score', fontsize=12)
    plt.title('BDE Score Comparison - Major Tech Stocks', fontsize=14, fontweight='bold')
    plt.xlim(0, 100)
    
    # Add score labels
    for i, (bar, score) in enumerate(zip(bars, df['BDE Score'])):
        plt.text(score + 1, bar.get_y() + bar.get_height()/2, 
                f'{score:.1f}', va='center', fontsize=10)
    
    plt.tight_layout()
    plt.show()
    
    # Insights
    print("\n💡 Insights:")
    best_stock = df.iloc[0]
    print(f"• Highest Score: {best_stock['Stock']} ({best_stock['BDE Score']:.1f})")
    print(f"• Average Score: {df['BDE Score'].mean():.1f}")
    print(f"• Average Confidence: {df['Confidence'].mean():.2%}")
else:
    print("\n⚠️  API unavailable. In a real scenario, you would see comparative analysis here.")

## 3. Deep Dive: Factor Analysis

Let's examine the factor breakdown for a single stock to understand how the score is calculated.

In [ ]:
# Deep analysis with full explanation
print("🔍 Deep analysis of Tencent (HK.00700)...\n")
detailed_result = analyze_stock("HK.00700", include_explanation=True)

if "error" not in detailed_result and 'explanation' in detailed_result:
    factors = detailed_result['explanation']['factor_breakdown']
    
    # Create pie chart
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Pie chart
    colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6']
    ax1.pie(factors.values(), labels=factors.keys(), autopct='%1.1f%%', 
           colors=colors, startangle=90)
    ax1.set_title('Factor Contribution Distribution', fontsize=12, fontweight='bold')
    
    # Bar chart
    bars = ax2.bar(factors.keys(), factors.values(), color=colors)
    ax2.set_ylabel('Contribution', fontsize=12)
    ax2.set_title('Factor Contributions (Absolute)', fontsize=12, fontweight='bold')
    ax2.set_ylim(0, max(factors.values()) * 1.2)
    
    # Add value labels
    for bar in bars:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.1f}', ha='center', va='bottom', fontsize=10)
    
    plt.tight_layout()
    plt.show()
    
    # Interpretation
    print("\n📊 Interpretation:")
    max_factor = max(factors, key=factors.get)
    min_factor = min(factors, key=factors.get)
    print(f"• Strongest Factor: {max_factor} ({factors[max_factor]:.1f})")
    print(f"• Weakest Factor: {min_factor} ({factors[min_factor]:.1f})")
    print(f"• Total Score: {detailed_result['score']:.1f}/100")
    print(f"• Confidence: {detailed_result['confidence']:.2%}")
    
else:
    print("⚠️  API unavailable. This demonstrates the factor analysis workflow.")
    print("\n💡 In production, you would see:")
    print("  • Technical indicators (RSI, MACD, Moving Averages)")
    print("  • Fundamental metrics (P/E, P/B, ROE)")
    print("  • Sentiment analysis (News, Social Media)")
    print("  • Risk assessment (Volatility, Liquidity)")
    print("  • Compliance metrics (Regulatory, ESG)")

## 4. EU AI Act Compliance Features

BDE Score is designed with EU AI Act compliance in mind. Let's examine the compliance features.

In [ ]:
print("🇪🇺 EU AI Act Compliance Features\n")
print("=" * 60)

compliance_features = {
    "Transparency": {
        "description": "Every decision includes detailed explanation",
        "implementation": "Factor breakdown, data sources, limitations"
    },
    "Traceability": {
        "description": "Full audit trail for all analysis requests",
        "implementation": "Request ID, timestamp, data snapshot"
    },
    "Human Oversight": {
        "description": "Automatic flagging for extreme scores",
        "implementation": "Score > 80 or < 20 triggers human review"
    },
    "Robustness": {
        "description": "Graceful degradation under uncertainty",
        "implementation": "Confidence scoring, risk warnings"
    }
}

for i, (feature, details) in enumerate(compliance_features.items(), 1):
    print(f"\n{i}. {feature}")
    print(f"   {details['description']}")
    print(f"   → {details['implementation']}")

print("\n" + "=" * 60)
print("\n💡 Why This Matters:")
print("Financial AI systems are classified as 'high-risk' under EU AI Act.")
print("BDE Score is designed from the ground up to meet these requirements.")

## 5. Advanced: Custom Weight Configuration

BDE Score allows customization of factor weights based on your investment strategy.

In [ ]:
# Example: Different investment strategies
strategies = {
    "Conservative": {
        "technical": 0.15,
        "fundamental": 0.35,
        "sentiment": 0.10,
        "risk": 0.30,
        "compliance": 0.10
    },
    "Aggressive": {
        "technical": 0.35,
        "fundamental": 0.15,
        "sentiment": 0.30,
        "risk": 0.10,
        "compliance": 0.10
    },
    "Balanced": {
        "technical": 0.25,
        "fundamental": 0.25,
        "sentiment": 0.20,
        "risk": 0.15,
        "compliance": 0.15
    }
}

# Visualize strategies
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (strategy_name, weights) in zip(axes, strategies.items()):
    factors = list(weights.keys())
    values = list(weights.values())
    
    bars = ax.bar(factors, values, color=['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6'])
    ax.set_ylim(0, 0.4)
    ax.set_title(f'{strategy_name} Strategy', fontsize=12, fontweight='bold')
    ax.set_ylabel('Weight')
    
    # Add value labels
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.0%}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print("\n💡 Strategy Selection Guide:")
print("• Conservative: Focus on fundamentals and risk management")
print("• Aggressive: Emphasize technical indicators and market sentiment")
print("• Balanced: Equal weight across all factors")

## 6. Next Steps

### 🚀 Get Started with BDE Score

1. **GitHub Repository**: [github.com/hbhqq9/bde-score](https://github.com/hbhqq9/bde-score)
2. **Documentation**: [hbhqq9.github.io/bde-score](https://hbhqq9.github.io/bde-score/)
3. **API Endpoint**: Available via Cloudflare Tunnel

### 📚 Learn More

- **Technical Deep Dive**: Read our [HackerNoon article](https://hackernoon.com/) (coming soon)
- **Chinese Tech Communities**: Articles on 掘金 and CSDN (coming soon)
- **GitHub Discussions**: Join the conversation in our community forums

### 🤝 Contribute

BDE Score is open source (MIT License). We welcome contributions:
- New data source integrations
- Additional technical indicators
- EU AI Act compliance enhancements
- Multi-language support

### ⭐ Star Us on GitHub

If you find BDE Score useful, please consider starring the repository to support the project!

---

**Disclaimer**: This tool is for technical research and educational purposes only. It does not constitute investment advice. Investment involves risks. Please make谨慎 decisions.